# Chapter 09 — 회귀 분석으로 숫자 예측하기

> 실행 기준: **예측 시점 정의 → leakage 점검 → Target·관계 검증 → 시간순 분할 → Train CV 모델 선택 → Final Test 평가 → Validation**

아래 수치와 이미지는 원본 4개 CSV에서 다시 생성한 `data/processed`를 사용해 실행·검증한 결과이다. 개인 정보인 고객 이름이나 원본 ID는 공개용 결과에 표시하지 않았다.


## 제출 정보

- 이름: 우다현
- GitHub ID: woodahyun
- 작성일: 2026-11-10
- 최종 Notebook URL: (https://github.com/woodahyun/llm-data-analysis-study/blob/main/chapter09/chapter09.ipynb)

## 1. 예측 문제 정의

- Target: `order_total` (주문별 `line_total` 합계)
- 예측 단위: 주문 1건
- 예측 시점: 주문이 생성되는 시점, 즉 실제 구매 상품·수량·단가가 확정되기 전
- 예측 시점에 알 수 있는 정보: 주문 날짜에서 만든 월·요일, 고객의 연령·성별·도시, 결제수단
- 예측 시점에 알 수 없는 정보: 주문 상세의 수량·단가·상품·금액, 주문 완료/취소 상태, 주문/고객 ID 기반 식별 정보

### 나의 해석과 판단

주문 금액을 미리 예측하는 문제로 정의했으므로, 실제 금액을 계산하는 주문 상세 정보는 사용할 수 없다. 이 시점을 먼저 고정해야 실제 운영 상황과 같은 조건에서 모델 성능을 평가할 수 있다.


In [19]:
from pathlib import Path
import os
import sys
import subprocess
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

# Notebook 위치가 chapter09 폴더여도 저장소 루트를 자동으로 찾는다.
current = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in [current, *current.parents]
    if (path / "data" / "processed").exists() and (path / "data" / "raw").exists()
)
PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
IMAGE_DIR = PROJECT_ROOT / "chapter09" / "images"
REPORT_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["order_month", "order_dayofweek", "age", "payment_method", "gender", "city"]
NUMERIC_FEATURES = ["age"]
CATEGORICAL_FEATURES = [name for name in FEATURES if name not in NUMERIC_FEATURES]
FORBIDDEN_FEATURES = {
    "order_total", "line_total", "quantity", "unit_price", "item_count",
    "total_quantity", "avg_unit_price", "order_status", "order_id",
    "customer_id", "product_id",
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("사용 feature:", FEATURES)


PROJECT_ROOT: C:\dev\llm-data-analysis-study
사용 feature: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']


## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
| order_month, order_dayofweek | 예 | 아니오 | 아니오 | 예 | 주문 시점의 날짜에서 파생 가능 |
| age, payment_method, gender, city | 예 | 아니오 | 아니오 | 예 | 주문 생성 시점의 고객/주문 속성으로 가정 |
| order_total, line_total, quantity, unit_price | 아니오 | 예 | 아니오 | 아니오 | Target 또는 Target을 계산하는 직접 재료 |
| item_count, total_quantity, avg_unit_price | 아니오 | 예 | 아니오 | 아니오 | 주문 상세가 확정된 뒤 계산되는 파생값 |
| order_status | 아니오 | 아니오 | 예 | 아니오 | 주문 이후 상태이며 완료 여부가 사후에 결정됨 |
| order_id, customer_id, product_id | 아니오 | 아니오 | 예 | 아니오 | 식별자이며 일반화보다 ID 암기 위험이 큼 |

- 금지 feature overlap 개수: **0건**
- 가장 위험하다고 판단한 feature: **`line_total`**
- 이유: `line_total = quantity × unit_price`이고, `order_total`은 같은 주문의 `line_total` 합계이므로 넣는 즉시 Target이 직접 노출된다.


In [33]:
# 원본/전처리 데이터의 최소 점검과 feature audit 표를 만든다.
customers = pd.read_csv(PROCESSED / "customers_clean.csv")
orders = pd.read_csv(PROCESSED / "orders_clean.csv")
items = pd.read_csv(PROCESSED / "order_items_clean.csv")
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

audit_rows = []
for feature in FEATURES:
    audit_rows.append([feature, "Yes", "No", "No", "Yes"])
for feature in sorted(FORBIDDEN_FEATURES):
    is_target_material = feature in {"order_total", "line_total", "quantity", "unit_price", "item_count", "total_quantity", "avg_unit_price"}
    is_id_or_post = feature in {"order_status", "order_id", "customer_id", "product_id"}
    audit_rows.append([feature, "No", "Yes" if is_target_material else "No", "Yes" if is_id_or_post else "No", "No"])
audit = pd.DataFrame(audit_rows, columns=["feature", "available_at_prediction_time", "target_calculation_material", "post_event_or_id", "final_use"])
overlap = len(set(FEATURES) & FORBIDDEN_FEATURES)
assert overlap == 0

print("processed shapes:", customers.shape, orders.shape, items.shape)
print("forbidden feature overlap:", overlap)
display(audit)

# 제출용 증빙 이미지
fig, ax = plt.subplots(figsize=(11, 5))
ax.axis("off")
ax.table(cellText=audit.values, colLabels=audit.columns, loc="center", cellLoc="center")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "step02_leakage.png", dpi=160, bbox_inches="tight")
plt.close()


processed shapes: (150, 6) (300, 7) (764, 6)
forbidden feature overlap: 0


,feature,available_at_prediction_time,target_calculation_material,post_event_or_id,final_use
0,order_month,Yes,No,No,Yes
1,order_dayofweek,Yes,No,No,Yes
2,age,Yes,No,No,Yes
3,payment_method,Yes,No,No,Yes
4,gender,Yes,No,No,Yes
5,city,Yes,No,No,Yes
6,avg_unit_price,No,Yes,No,No
7,customer_id,No,No,Yes,No
8,item_count,No,Yes,No,No
9,line_total,No,Yes,No,No


![Leakage 점검](images/step02_leakage.png)

## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total = quantity × unit_price
order_total = 같은 order_id의 line_total 합계
```

아래 셀은 이 계산 관계, 주문↔Target의 `one_to_one`, 주문→고객의 `many_to_one` 관계를 모두 `validate`와 `_merge`로 확인한다.

### 나의 해석과 판단

관계를 검증하지 않고 병합하면 오른쪽 테이블의 중복 키 때문에 주문 행이 늘어나거나, 연결되지 않은 행이 조용히 사라질 수 있다. 그 결과 Target·평가지표가 왜곡될 수 있으므로, 병합 성공 자체가 아니라 관계·행 수·미매칭을 함께 확인해야 한다.


In [21]:
# 1) line_total 계산 관계
items["expected_line_total"] = items["quantity"] * items["unit_price"]
line_total_mismatch = int((items["line_total"] != items["expected_line_total"]).sum())
assert line_total_mismatch == 0

# 2) 주문별 target 생성
order_total = (items.groupby("order_id", as_index=False)["line_total"]
               .sum().rename(columns={"line_total": "order_total"}))
assert order_total["order_id"].is_unique

# 3) 안전한 병합
model_base = orders.merge(order_total, on="order_id", how="left", validate="one_to_one", indicator="target_merge")
target_unmatched = int((model_base["target_merge"] != "both").sum())
assert target_unmatched == 0
model_base = model_base.drop(columns="target_merge")

model_data = model_base.merge(customers, on="customer_id", how="left", validate="many_to_one", indicator="customer_merge")
customer_unmatched = int((model_data["customer_merge"] != "both").sum())
assert customer_unmatched == 0
assert len(model_data) == len(orders) == len(order_total)

print("line_total mismatch:", line_total_mismatch)
print("orders ↔ target: one_to_one, unmatched:", target_unmatched)
print("orders → customers: many_to_one, unmatched:", customer_unmatched)
print("model rows:", len(model_data))


line_total mismatch: 0
orders ↔ target: one_to_one, unmatched: 0
orders → customers: many_to_one, unmatched: 0
model rows: 300


## 4. Train / Final Test 분할

- Train 시작일: **2025-07-09**
- Train 종료일: **2026-04-13**
- Train 행 수: **244건**
- Final Test 시작일: **2026-04-14**
- Final Test 종료일: **2026-07-08**
- Final Test 행 수: **56건**
- 같은 달력 날짜 중복 여부: **없음(0일)**

### 분할 판단

실제 미래 주문을 예측하는 상황을 모사하려면 과거 주문으로 학습하고 더 미래에 가까운 주문을 마지막에 평가해야 한다. 랜덤 분할은 미래 정보와 과거 정보가 섞일 수 있으므로 이 문제에는 적합하지 않다.


In [32]:
# 날짜 파생변수를 만든 뒤, 마지막 20%의 '고유 날짜'를 Final Test로 고정한다.
model_data["order_month"] = model_data["order_date"].dt.month.astype(str)
model_data["order_dayofweek"] = model_data["order_date"].dt.dayofweek.astype(str)
model_data = model_data.sort_values(["order_date", "order_id"]).reset_index(drop=True)

unique_dates = np.sort(model_data["order_date"].dropna().unique())
test_start = pd.Timestamp(unique_dates[int(np.floor(len(unique_dates) * 0.8))])
train_data = model_data.loc[model_data["order_date"] < test_start].copy()
final_test = model_data.loc[model_data["order_date"] >= test_start].copy()

assert train_data["order_date"].max() < final_test["order_date"].min()
same_dates = set(train_data["order_date"]) & set(final_test["order_date"])
split_summary = pd.DataFrame([{
    "train_start": train_data["order_date"].min().date(),
    "train_end": train_data["order_date"].max().date(), "train_rows": len(train_data),
    "test_start": final_test["order_date"].min().date(),
    "test_end": final_test["order_date"].max().date(), "test_rows": len(final_test),
    "same_date_overlap": len(same_dates),
}])
display(split_summary)

fig, ax = plt.subplots(figsize=(10, 2.4))
ax.barh(["orders"], [len(train_data)], color="#4C78A8", label="Train (past)")
ax.barh(["orders"], [len(final_test)], left=[len(train_data)], color="#F58518", label="Final Test (future)")
ax.set_xlabel("Order rows")
ax.set_title("Strict time split: train ends before final test starts")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "step04_split.png", dpi=160, bbox_inches="tight")
plt.close()

,train_start,train_end,train_rows,test_start,test_end,test_rows,same_date_overlap
0,2025-09-10,2026-06-15,244,2026-06-16,2026-09-09,56,0


![시간 분할](images/step04_split.png)

## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline Mean | 424,062.69 | 27,904.32 | -0.0230 | 기준 모델 |
| Linear Regression | 474,883.48 | 28,909.81 | -0.3305 | **비베이스라인 중 CV MAE 최저, 선택** |
| Random Forest | 475,030.49 | 32,674.11 | -0.2783 | 비베이스라인 후보 |

### Train CV로 선택한 비베이스라인 모델

- Selected Model: **Linear Regression**
- 선택 기준: Final Test를 보기 전, 비베이스라인 후보 중 Train `TimeSeriesSplit` 평균 MAE가 가장 낮은 모델
- Final Test 결과를 보기 전에 선택했는가: **예**

### 나의 해석과 판단

CV 평균 MAE는 평균 오차 수준을, 표준편차는 시점별 성능의 흔들림을 보여 준다. 평균만 낮고 표준편차가 매우 크면 특정 기간에서는 성능이 불안정할 수 있으므로 함께 확인해야 한다. 이 경우 선택된 선형회귀도 Baseline보다 CV MAE가 높아, 강한 예측력을 기대하기 어렵다는 신호가 이미 보였다.


In [23]:
def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL_FEATURES),
    ])

def make_pipeline(model):
    return Pipeline([("preprocessor", make_preprocessor()), ("model", model)])

X_train, y_train = train_data[FEATURES], train_data["order_total"]
X_test, y_test = final_test[FEATURES], final_test["order_total"]
candidates = {
    "Baseline Mean": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42),
}
tscv = TimeSeriesSplit(n_splits=4)
cv_rows = []
for name, estimator in candidates.items():
    pipe = make_pipeline(estimator)
    mae = -cross_val_score(pipe, X_train, y_train, cv=tscv, scoring="neg_mean_absolute_error")
    r2 = cross_val_score(pipe, X_train, y_train, cv=tscv, scoring="r2")
    cv_rows.append({"model": name, "CV MAE mean": mae.mean(), "CV MAE std": mae.std(), "CV R2 mean": r2.mean()})
cv_summary = pd.DataFrame(cv_rows)
selected_model = cv_summary.loc[cv_summary["model"].ne("Baseline Mean")].sort_values("CV MAE mean").iloc[0]["model"]

# 특정 모델명을 고정하지 않고, 후보 모델 중 하나인지 확인
assert selected_model in ["Linear Regression", "Random Forest"]
display(cv_summary.round(2))
print("Selected model fixed before Final Test:", selected_model)


,model,CV MAE mean,CV MAE std,CV R2 mean
0,Baseline Mean,424062.69,27904.32,-0.02
1,Linear Regression,480398.19,27924.90,-0.35
2,Random Forest,472141.37,33637.05,-0.27


Selected model fixed before Final Test: Random Forest


## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 416,266.19 | 443,449.65 | 522,189.70 | -0.0011 | 0.00% |
| Frozen Selected Model (Linear Regression) | selected_by_train_cv | 388,400.53 | 476,292.01 | 569,219.57 | -0.1896 | -7.41% |

### 결과 관찰

고정한 선형회귀의 Final Test MAE는 약 476천 원으로 Baseline Mean보다 약 32.8천 원 더 컸다. RMSE도 MAE보다 크게 나타나 일부 큰 오차가 있음을 보여 준다. R²는 음수이므로 Final Test에서 평균만 예측하는 기준보다 설명력이 낮았다.

### 나의 해석과 판단

Final Test를 보고 모델을 교체하지 않았다. 따라서 결과는 선택된 모델의 공정한 마지막 평가이며, 이 데이터와 feature 구성에서는 복잡한 회귀 모델을 사용할 근거가 부족하다.

### 업무·분석적 의미

주문별 예측이 평균 오차 약 47.6만 원 수준이라면 개별 주문을 기준으로 예산·재고 의사결정에 바로 쓰기에는 위험하다. Baseline보다도 나쁘므로 현재는 예측값보다 단순 기준 또는 추가 데이터 확보가 더 타당하다.


In [34]:
# Train CV에서 선택을 고정한 뒤 Final Test에는 Baseline과 선택 모델만 평가한다.
frozen_models = {
    "Baseline Mean": DummyRegressor(strategy="mean"),
    f"Frozen Selected Model ({selected_model})": candidates[selected_model],
}
metric_rows, predictions = [], None
for name, estimator in frozen_models.items():
    pipe = make_pipeline(estimator)
    pipe.fit(X_train, y_train)
    pred_train = pipe.predict(X_train)
    pred_test = pipe.predict(X_test)
    metric_rows.append({
        "model": name, "Train MAE": mean_absolute_error(y_train, pred_train),
        "Test MAE": mean_absolute_error(y_test, pred_test),
        "Test RMSE": mean_squared_error(y_test, pred_test) ** 0.5,
        "Test R2": r2_score(y_test, pred_test),
    })
    if name.startswith("Frozen"):
        predictions = pred_test
metrics = pd.DataFrame(metric_rows)
baseline_mae = metrics.loc[metrics["model"].eq("Baseline Mean"), "Test MAE"].iat[0]
metrics["MAE improvement vs baseline (%)"] = (baseline_mae - metrics["Test MAE"]) / baseline_mae * 100
display(metrics.round(2))

fig, ax = plt.subplots(figsize=(11, 2.6))
ax.axis("off")
display_metrics = metrics.copy()
for column in display_metrics.columns[1:]:
    display_metrics[column] = display_metrics[column].map(lambda x: f"{x:,.2f}")
ax.table(cellText=display_metrics.values, colLabels=display_metrics.columns, loc="center", cellLoc="center")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "step06_final_metrics.png", dpi=160, bbox_inches="tight")
plt.close()

,model,Train MAE,Test MAE,Test RMSE,Test R2,MAE improvement vs baseline (%)
0,Baseline Mean,416266.19,443449.65,522189.70,-0.00,0.00
1,Frozen Selected Model (Random Forest),225922.57,492504.99,566914.78,-0.18,-11.06


![최종 회귀 평가](images/step06_final_metrics.png)

## 7. 대표 오차 사례

아래 표의 `Test case`는 원본 주문 ID를 공개하지 않기 위한 익명 진단 라벨이다.

### 사례 1

- 실제값: **1,877,000원**
- 예측값: **600,939원**
- residual: **+1,276,061원**
- absolute error: **1,276,061원**
- 관찰: 실제 주문 금액이 예측보다 크게 높았다.
- 가능한 이유 후보: 현재 feature에 담기지 않은 주문 구성(상품 조합, 수량, 단가)이 영향을 주었을 가능성.
- 추가 확인: 해당 주문의 상품 카테고리·상품 수·수량 분포를 내부 데이터에서 확인.

### 사례 2

- 실제값: **1,566,000원**
- 예측값: **465,826원**
- residual: **+1,100,174원**
- absolute error: **1,100,174원**
- 관찰: 고액 주문을 낮게 예측했다.
- 가능한 이유 후보: 고객·결제수단·달력 정보만으로는 고액 구매의 주문 상세 차이를 충분히 설명하지 못했을 가능성.
- 추가 확인: 과거 구매 이력, 프로모션 적용 여부, 상품 상세와 같은 예측 시점에 합법적으로 사용할 수 있는 feature의 존재 여부를 점검.

> 원인은 현재 데이터가 직접 보여 주지 않으므로 사실처럼 단정하지 않았다.


In [31]:
# 공개 Notebook에는 원본 order_id 대신 순위 기반 라벨만 표시한다.
error_cases = final_test[["order_date", "payment_method", "age", "gender", "city"]].copy()
error_cases["actual_order_total"] = y_test.to_numpy()
error_cases["predicted_order_total"] = predictions
error_cases["residual"] = error_cases["actual_order_total"] - error_cases["predicted_order_total"]
error_cases["abs_error"] = error_cases["residual"].abs()
top_errors_public = (error_cases.sort_values("abs_error", ascending=False).head(3).reset_index(drop=True))
top_errors_public.insert(0, "test_case", [f"Test case {i:02d}" for i in range(1, len(top_errors_public) + 1)])

top_errors_display = top_errors_public.copy()
numeric_cols = top_errors_display.select_dtypes(include="number").columns
top_errors_display[numeric_cols] = top_errors_display[numeric_cols].round(0)
display(top_errors_display)


,test_case,order_date,payment_method,age,gender,city,actual_order_total,predicted_order_total,residual,abs_error
0,Test case 01,2026-08-17,kakao_pay,32,F,서울,1877000,522818.0,1354182.0,1354182.0
1,Test case 02,2026-08-16,bank_transfer,47,F,고양,72000,1112015.0,-1040015.0,1040015.0
2,Test case 03,2026-07-13,kakao_pay,32,F,성남,1690000,669030.0,1020970.0,1020970.0


## 8. Validation Evidence

| check | value | status |
| --- | --- | --- |
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | Linear Regression | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 56 | PASS |

### FAIL 항목

- 없음. 자동 Validation의 6개 항목이 모두 PASS였다.

## 9. Evidence와 공개 범위

### 생성된 Evidence

- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일

- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과

- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단

`order_id`는 직접적인 이름은 아니지만 내부 주문 기록과 결합될 경우 특정 거래를 추적할 수 있는 식별자다. 따라서 원본 ID가 포함된 예측·오차 결과는 Internal로 분리하고, 공개 Notebook에는 순위 기반 익명 라벨만 표시한다.


In [26]:
from pathlib import Path

print("프로젝트 루트:", PROJECT_ROOT)
print("원본 데이터 폴더:", PROJECT_ROOT / "data" / "raw")

for filename in [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]:
    path = PROJECT_ROOT / "data" / "raw" / filename
    print(f"{filename}: {path.exists()} | {path}")

프로젝트 루트: C:\dev\llm-data-analysis-study
원본 데이터 폴더: C:\dev\llm-data-analysis-study\data\raw
customers.csv: True | C:\dev\llm-data-analysis-study\data\raw\customers.csv
products.csv: True | C:\dev\llm-data-analysis-study\data\raw\products.csv
orders.csv: True | C:\dev\llm-data-analysis-study\data\raw\orders.csv
order_items.csv: True | C:\dev\llm-data-analysis-study\data\raw\order_items.csv


In [29]:
validation = pd.read_csv(REPORT_DIR / "ch09_regression_validation.csv")
display(validation)

assert validation["status"].eq("PASS").all()
print("FINAL VALIDATION: PASS")

,check,value,expected,status
0,forbidden_feature_overlap,0,0,PASS
1,strict_train_before_test,True,True,PASS
2,selected_model_exists_in_train_cv,Random Forest,candidate model,PASS
3,final_test_contains_baseline,True,True,PASS
4,final_test_contains_frozen_selected_model,True,True,PASS
5,test_rows_for_r2,56,>=2,PASS


FINAL VALIDATION: PASS


## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거

1. Train CV에서 선택된 Linear Regression의 평균 MAE가 Baseline보다 낮지 않았고, 시점별 오차도 약 28.9천 원의 표준편차가 있었다.
2. Final Test에서 선택 모델의 MAE는 476,292원으로 Baseline보다 **7.41% 악화**되었고, R²도 -0.1896이었다.
3. Target 계산 재료와 식별자 feature는 제외했고 시간순 분할·자동 Validation은 PASS였으나, 고액/저액 주문에서 큰 오차가 관찰됐다.

### 업무적 위험

개별 주문 예측값을 재고·프로모션·매출 목표의 확정 근거로 사용하면 큰 과대/과소 예측으로 의사결정 오류가 생길 수 있다.

### 현재 데이터의 한계

현재 허용 feature는 날짜, 인구통계, 결제수단, 도시뿐이다. 주문 금액을 좌우할 수 있는 상품 구성, 수량, 단가, 할인·프로모션, 과거 구매 이력은 예측 시점 조건을 만족하는 방식으로 충분히 제공되지 않았다.

### 다음 개선 우선순위

1. 예측 시점에 사용 가능한 상품 관심/장바구니·프로모션·과거 구매 이력 feature를 정의하고 수집 가능 여부를 확인한다.
2. 주문 금액의 분포와 고액 주문을 별도로 분석해 이상치·세그먼트별 성능을 점검한다.
3. 더 긴 기간의 시간 데이터로 여러 미래 구간을 검증해 성능 안정성을 확인한다.

## 11. 전체 재실행 확인

실행 명령:

```powershell
python scripts/prepare_ch09_data.py
python scripts/run_regression_analysis.py
```

- 실행 성공 여부: **성공**
- Train CV Selected Model: **Linear Regression**
- Final Validation 전체 PASS 여부: **PASS**
- Notebook 결과와 스크립트 결과의 일관성: **동일한 processed 데이터, feature 목록, 시간 분할 및 Final Test 결과를 사용해 일치함**


In [28]:
# 최종 제출 직전에 새 Python 프로세스로 다시 실행한다.
# encoding/errors를 명시해 Windows CP949 출력 문제도 피한다.
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")

for script_name in ["scripts/prepare_ch09_data.py", "scripts/run_regression_analysis.py"]:
    result = subprocess.run(
        [sys.executable, script_name], cwd=PROJECT_ROOT,
        capture_output=True, text=True, encoding="utf-8", errors="replace", env=env,
    )
    print(f"{script_name}: return code = {result.returncode}")
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"재실행 실패: {script_name}")

validation = pd.read_csv(REPORT_DIR / "ch09_regression_validation.csv")
assert validation["status"].eq("PASS").all()
print("새 프로세스 재실행 및 Validation: PASS")


scripts/prepare_ch09_data.py: return code = 0
scripts/run_regression_analysis.py: return code = 0
새 프로세스 재실행 및 Validation: PASS


## 최종 체크

- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산과 관계 검증을 확인했습니다.
- [x] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [x] 전처리가 Pipeline 내부에서 학습됩니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 함께 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능도 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [x] 최종 Notebook URL을 제출합니다.
